<a href="https://colab.research.google.com/github/thanwalai-k03/DataScience-Assignment1/blob/main/%E0%B8%AA%E0%B8%B3%E0%B9%80%E0%B8%99%E0%B8%B2%E0%B8%82%E0%B8%AD%E0%B8%87_SC612104_Session8_Pandas_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SC 612 104 Essential Data Science
## คาบที่ 8: Series, DataFrame, read_csv, head/tail, info, describe, dtypes, loc, iloc, Boolean Indexing

**ผู้สอน:** อ.ดร.พิชญา วิรัชโชติเสถียร (Pitchaya Wiratchotisatian)
**ภาควิชาสถิติ คณะวิทยาศาสตร์ มหาวิทยาลัยขอนแก่น**

---

### เนื้อหาในคาบนี้
1. ทำไมต้องมี pandas
2. **Series** — โครงสร้างข้อมูล 1 มิติ
3. **DataFrame** — โครงสร้างข้อมูล 2 มิติ
4. `read_csv()` — โหลดข้อมูลจริงจากไฟล์
5. `head()` / `tail()` — ดูข้อมูลตัวอย่าง
6. `info()` — ภาพรวมโครงสร้างข้อมูล
7. `describe()` — สถิติพื้นฐาน
8. `dtypes` — ชนิดข้อมูลของแต่ละคอลัมน์
9. `loc` และ `iloc` — เลือกข้อมูลแบบระบุตำแหน่ง
10. **Boolean Indexing** — กรองข้อมูลด้วยเงื่อนไข
11. แบบฝึกหัดท้ายคาบ (ไม่มีเฉลย)

> 📁 **ไฟล์ที่ต้องใช้ในคาบนี้:** `world_weather_sample.csv` — ข้อมูลอากาศจำลองจาก 23 เมืองทั่วโลก 90 วัน (รวม 2,075 แถว) **ข้อมูลนี้สร้างขึ้นเพื่อการเรียนการสอนเท่านั้น ไม่ใช่ข้อมูลอากาศจริง** แต่จงใจให้มี missing values, ค่าผิดปกติ (outliers), แถวซ้ำ, และตัวพิมพ์ไม่สม่ำเสมอ เหมือนข้อมูลจริงที่ต้องทำความสะอาดก่อนใช้งาน — อัปโหลดไฟล์นี้เข้า Colab ก่อนเริ่ม (ลากไฟล์ลงในแถบ Files ทางซ้าย หรือใช้โค้ดอัปโหลดด้านล่าง)


In [ ]:
# ถ้าใช้ Google Colab และยังไม่ได้อัปโหลดไฟล์ ให้รันเซลล์นี้เพื่ออัปโหลด world_weather_sample.csv
# (ถ้ารันใน Jupyter บนเครื่องตัวเอง และไฟล์อยู่โฟลเดอร์เดียวกันแล้ว ไม่ต้องรันเซลล์นี้)

try:
    from google.colab import files
    uploaded = files.upload()   # จะมีปุ่มให้เลือกไฟล์จากเครื่อง - เลือก world_weather_sample.csv
except ImportError:
    print("ไม่ได้รันบน Colab - ข้ามขั้นตอนนี้ (ตรวจสอบว่าไฟล์ CSV อยู่ในโฟลเดอร์เดียวกับ Notebook นี้)")

KeyboardInterrupt: 

---
## 1. ทำไมต้องมี pandas?

จากคาบที่แล้ว NumPy array ทำให้คำนวณกับข้อมูลตัวเลขเร็วขึ้นมาก แต่ข้อมูลจริงมักมีลักษณะ:
- มีหลายคอลัมน์ที่**ชนิดข้อมูลต่างกัน** (ชื่อเมืองเป็น text, อุณหภูมิเป็น float, วันที่เป็น date)
- มี**ชื่อคอลัมน์**กำกับความหมาย (ไม่ใช่แค่ index ตัวเลขแบบ NumPy array)
- ต้องอ่านจากไฟล์ (CSV, Excel, ฐานข้อมูล) และมักมี**ข้อมูลขาดหาย/ผิดปกติ**ต้องจัดการ

**pandas** คือไลบรารีที่ออกแบบมาสำหรับงานแบบนี้โดยเฉพาะ — เป็นเครื่องมือที่ใช้กันมากที่สุดในการทำ Data Science ด้วย Python ตั้งแต่ขั้นโหลดข้อมูล ทำความสะอาด วิเคราะห์ ไปจนถึงเตรียมข้อมูลก่อนเข้าโมเดล

pandas มี 2 โครงสร้างข้อมูลหลักที่ต้องรู้จัก: **Series** (1 มิติ) และ **DataFrame** (2 มิติ)


In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)

2.2.2


---
## 2. Series — โครงสร้างข้อมูล 1 มิติ

**Series** คือ "คอลัมน์เดียว" ของข้อมูล — คล้าย NumPy array แต่มี **index** (ป้ายชื่อกำกับแต่ละค่า) ติดมาด้วย เปรียบเทียบกับ Collections ที่เรียนมา: Series มีลักษณะคล้าย **dict** (มี key/index จับคู่กับ value) ผสมกับ **list/array** (มีลำดับ, ทำ vectorized operation ได้)


In [ ]:
# สร้าง Series จาก list
temperatures = pd.Series([28.5, 30.2, 27.8, 29.1, 31.0])
print(temperatures)
print(type(temperatures))

0    28.5
1    30.2
2    27.8
3    29.1
4    31.0
dtype: float64
<class 'pandas.core.series.Series'>


สังเกตว่าทางซ้ายมีเลข 0, 1, 2, 3, 4 — นี่คือ **index** ที่ pandas สร้างให้อัตโนมัติ (เหมือน index ของ list) แต่กำหนด index เองได้ ทำให้ Series ทำงานคล้าย **dict** มาก


In [ ]:
# สร้าง Series พร้อมกำหนด index เอง (คล้าย dict ที่เรียนมา - key คือ index)
city_temps = pd.Series(
    [30.5, 16.2, 28.0, 11.5],
    index=["Bangkok", "Tokyo", "Singapore", "London"]
)
print(city_temps)

# เข้าถึงค่าผ่าน index แบบ dict
print("\nอุณหภูมิที่ Tokyo:", city_temps["Tokyo"])

# หรือเข้าถึงผ่านตำแหน่งแบบ list/array ก็ได้ (positional)
print("ค่าตำแหน่งที่ 0:", city_temps.iloc[0])

Bangkok      30.5
Tokyo        16.2
Singapore    28.0
London       11.5
dtype: float64

อุณหภูมิที่ Tokyo: 16.2
ค่าตำแหน่งที่ 0: 30.5


### 2.1 สร้าง Series จาก dict โดยตรง

วิธีนี้ใช้บ่อยมาก เพราะ key ของ dict จะกลายเป็น index ของ Series โดยอัตโนมัติ


In [ ]:
city_temps_dict = {"Bangkok": 30.5, "Tokyo": 16.2, "Singapore": 28.0, "London": 11.5}
city_temps_series = pd.Series(city_temps_dict)
print(city_temps_series)
print(type(city_temps_series))

Bangkok      30.5
Tokyo        16.2
Singapore    28.0
London       11.5
dtype: float64
<class 'pandas.core.series.Series'>


### 2.2 Vectorized Operation บน Series

Series ทำ vectorized operation ได้เหมือน NumPy array ทุกอย่าง เพราะ pandas สร้างอยู่บน NumPy


In [ ]:
city_temps = pd.Series([30.5, 16.2, 28.0, 11.5], index=["Bangkok", "Tokyo", "Singapore", "London"])

print("แปลงเป็นฟาเรนไฮต์:")
print(city_temps * 9/5 + 32)

print("\nเมืองที่อุณหภูมิ > 20:")
print(city_temps[city_temps > 20])   # boolean indexing แบบเดียวกับ NumPy array

print("\nค่าเฉลี่ย:", city_temps.mean())
print("ค่าสูงสุด:", city_temps.max())

แปลงเป็นฟาเรนไฮต์:
Bangkok      86.90
Tokyo        61.16
Singapore    82.40
London       52.70
dtype: float64

เมืองที่อุณหภูมิ > 20:
Bangkok      30.5
Singapore    28.0
dtype: float64

ค่าเฉลี่ย: 21.55
ค่าสูงสุด: 30.5


---
## 3. DataFrame — โครงสร้างข้อมูล 2 มิติ

**DataFrame** คือ "ตาราง" ข้อมูล — เปรียบเสมือนตาราง Excel หรือฐานข้อมูล ที่มีหลายคอลัมน์ แต่ละคอลัมน์คือ **Series** หนึ่งตัว ทุกคอลัมน์ใน DataFrame เดียวกันใช้ **index** ร่วมกัน

DataFrame เปรียบเทียบกับ Collections ที่เรียนมา: คล้าย **list ของ dict** (แต่ละแถวคือ 1 record) หรือ **dict ของ list** (แต่ละคอลัมน์คือ 1 list ของค่า) — pandas รองรับสร้างได้ทั้ง 2 แบบ


In [ ]:
# สร้าง DataFrame จาก dict ของ list (คอลัมน์: ชื่อคอลัมน์ -> list ของค่า)
data = {
    "city": ["Bangkok", "Tokyo", "Singapore", "London"],
    "temperature_c": [30.5, 16.2, 28.0, 11.5],
    "humidity_pct": [75, 60, 80, 78],
}

df = pd.DataFrame(data)
print(df)
print(type(df))

        city  temperature_c  humidity_pct
0    Bangkok           30.5            75
1      Tokyo           16.2            60
2  Singapore           28.0            80
3     London           11.5            78
<class 'pandas.core.frame.DataFrame'>


### 3.1 สร้าง DataFrame จาก list ของ dict (เชื่อมกับ Collections ที่เรียนมา)

อีกวิธีที่ใช้บ่อย เหมาะกับข้อมูลที่มาเป็น "รายการของ record" เช่น ข้อมูลจาก API ที่มักเป็น list ของ dict


In [ ]:
records = [
    {"city": "Bangkok", "temperature_c": 30.5, "humidity_pct": 75},
    {"city": "Tokyo", "temperature_c": 16.2, "humidity_pct": 60},
    {"city": "Singapore", "temperature_c": 28.0, "humidity_pct": 80},
    {"city": "London", "temperature_c": 11.5, "humidity_pct": 78},
]

df_v2 = pd.DataFrame(records)
print(df_v2)

# ตรวจสอบว่าได้ผลลัพธ์เหมือนกันทั้ง 2 วิธี
print("\nเหมือนกันไหม:", df.equals(df_v2))

### 3.2 โครงสร้างพื้นฐานของ DataFrame

| ส่วนประกอบ | ความหมาย |
|---|---|
| **columns** | ชื่อคอลัมน์ทั้งหมด (เข้าถึงด้วย `.columns`) |
| **index** | ป้ายชื่อกำกับแต่ละแถว (เข้าถึงด้วย `.index`) |
| **values** | ข้อมูลทั้งหมดในรูปแบบ NumPy array (เข้าถึงด้วย `.values`) |
| **เลือกคอลัมน์เดียว** | `df["ชื่อคอลัมน์"]` จะได้ **Series** กลับมา |
| **เลือกหลายคอลัมน์** | `df[["คอลัมน์1", "คอลัมน์2"]]` จะได้ **DataFrame** กลับมา (สังเกตวงเล็บ `[]` 2 ชั้น) |


In [ ]:
print("columns:", df.columns.tolist())
print("index:", df.index.tolist())
print()

# เลือกคอลัมน์เดียว -> ได้ Series
city_column = df["city"]
print(type(city_column))
print(city_column)

print()

# เลือกหลายคอลัมน์ -> ได้ DataFrame (สังเกต [[ ]] สองชั้น)
subset = df[["city", "temperature_c"]]
print(type(subset))
print(subset)

columns: ['city', 'temperature_c', 'humidity_pct']
index: [0, 1, 2, 3]

<class 'pandas.core.series.Series'>
0      Bangkok
1        Tokyo
2    Singapore
3       London
Name: city, dtype: object

<class 'pandas.core.frame.DataFrame'>
        city  temperature_c
0    Bangkok           30.5
1      Tokyo           16.2
2  Singapore           28.0
3     London           11.5


---
## 4. `read_csv()` — โหลดข้อมูลจริงจากไฟล์

จนถึงตอนนี้สร้าง DataFrame จากข้อมูลที่พิมพ์เอง — แต่ในงานจริง ข้อมูลมักมาจากไฟล์ **CSV** (Comma-Separated Values) ซึ่งเป็นรูปแบบไฟล์ที่ใช้กันแพร่หลายที่สุดในการแลกเปลี่ยนข้อมูลตาราง

`pd.read_csv("ชื่อไฟล์.csv")` เป็นคำสั่งที่ใช้บ่อยที่สุดในการเริ่มงาน Data Science ทุกโปรเจกต์


In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
!find / -name "world_weather_sample*.csv" 2>/dev/null

/world_weather_sample-3.csv


In [ ]:
df = pd.read_csv("/content/world_weather_sample-3.csv")
print("โหลดข้อมูลสำเร็จ! ขนาดของข้อมูล:", df.shape)   # (จำนวนแถว, จำนวนคอลัมน์)
df

โหลดข้อมูลสำเร็จ! ขนาดของข้อมูล: (2075, 10)


,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
0,983,2025-03-24,New York,USA,North America,15.7,72.6,6.3,15.9,1005.4
1,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1
2,1022,2025-02-01,Los Angeles,USA,North America,20.6,46.8,1.2,8.6,1013.5
3,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1
4,1834,2025-02-03,Cape Town,South Africa,Africa,19.1,55.0,0.0,16.6,1015.1
...,...,...,...,...,...,...,...,...,...,...
2070,1093,2025-01-13,Mexico City,Mexico,North America,17.8,63.3,7.4,10.5,1001.2
2071,1769,2025-02-28,Nairobi,Kenya,Africa,20.6,65.8,3.7,13.3,1015.6
2072,1738,2025-01-28,Nairobi,Kenya,Africa,21.7,51.0,9.5,27.2,1013.9
2073,1210,2025-02-09,Toronto,Canada,North America,NaN,65.0,9.1,6.9,1007.8


**สังเกต:** ตัวเลข `(2075, 10)` ที่ได้จาก `.shape` คือ **tuple** ที่เรียนมา — 2075 แถว, 10 คอลัมน์ และ Jupyter/Colab จะแสดงตารางให้สวยงามอัตโนมัติเมื่อพิมพ์ชื่อ DataFrame ไว้บรรทัดสุดท้ายของเซลล์ (ไม่ต้อง `print()`)

### 4.1 พารามิเตอร์ที่ใช้บ่อยของ `read_csv()`

| พารามิเตอร์ | ความหมาย |
|---|---|
| `sep` | ตัวคั่นคอลัมน์ (default คือ `,` แต่บางไฟล์ใช้ `;` หรือ Tab) |
| `encoding` | การเข้ารหัสตัวอักษร (เช่น `"utf-8"` สำหรับไฟล์ที่มีภาษาไทย) |
| `usecols` | ระบุว่าจะอ่านเฉพาะบางคอลัมน์ |
| `nrows` | จำกัดจำนวนแถวที่อ่าน (มีประโยชน์เวลาไฟล์ใหญ่มาก อยากดูตัวอย่างก่อน) |


In [ ]:
# ตัวอย่าง: อ่านแค่บางคอลัมน์ และจำกัดจำนวนแถว
df_preview = pd.read_csv(
    "/content/world_weather_sample-3.csv",
    usecols=["city", "temperature_c", "humidity_pct"],
    nrows=5
)
print(df_preview)

          city  temperature_c  humidity_pct
0     New York           15.7          72.6
1        Tokyo           18.3          60.6
2  Los Angeles           20.6          46.8
3       Mumbai           31.0          73.4
4    Cape Town           19.1          55.0


---
## 5. `head()` / `tail()` — ดูข้อมูลตัวอย่าง

เมื่อข้อมูลมีหลายพันแถว การ print ทั้งหมดไม่มีประโยชน์ — ใช้ `.head()` ดู "หัวตาราง" และ `.tail()` ดู "ท้ายตาราง" แทน


In [ ]:
df = pd.read_csv("/content/world_weather_sample-3.csv")

print("--- 5 แถวแรก (default) ---")
print(df.head())

print("\n--- 3 แถวแรก (ระบุจำนวนเอง) ---")
print(df.head(3))

print("\n--- 5 แถวสุดท้าย ---")
print(df.tail())

--- 5 แถวแรก (default) ---
   record_id        date         city       country         region  \
0        983  2025-03-24     New York           USA  North America   
1        220  2025-02-09        Tokyo         Japan           Asia   
2       1022  2025-02-01  Los Angeles           USA  North America   
3        469  2025-01-19       Mumbai         India           Asia   
4       1834  2025-02-03    Cape Town  South Africa         Africa   

   temperature_c  humidity_pct  rainfall_mm  wind_speed_kmh  pressure_hpa  
0           15.7          72.6          6.3            15.9        1005.4  
1           18.3          60.6          0.0            20.5        1015.1  
2           20.6          46.8          1.2             8.6        1013.5  
3           31.0          73.4          8.0            11.8        1000.1  
4           19.1          55.0          0.0            16.6        1015.1  

--- 3 แถวแรก (ระบุจำนวนเอง) ---
   record_id        date         city country         region  t

> 💡 **เทคนิคการสำรวจข้อมูลเบื้องต้น:** เวลาเปิดข้อมูลใหม่ครั้งแรก ควรรัน `.head()` เสมอ เพื่อดูว่าคอลัมน์มีอะไรบ้าง ข้อมูลหน้าตาเป็นแบบไหน ก่อนจะวิเคราะห์ต่อ — เป็นขั้นตอนที่ทำเป็นกิจวัตรในงานจริงทุกครั้ง


---
## 6. `info()` — ภาพรวมโครงสร้างข้อมูล

`.info()` ให้ภาพรวมทั้งหมดของ DataFrame ในคำสั่งเดียว: จำนวนแถว, ชื่อคอลัมน์, จำนวนค่าที่ไม่ใช่ null ในแต่ละคอลัมน์, และชนิดข้อมูล — เป็นคำสั่งที่ควรรันแทบทุกครั้งหลังโหลดข้อมูลใหม่


In [ ]:
df.info()

**อ่านผลลัพธ์ `.info()` อย่างไร:**
- `RangeIndex: 2075 entries` → มี 2075 แถว
- `Data columns (total 10 columns)` → มี 10 คอลัมน์
- คอลัมน์ `Non-Null Count` → จำนวนค่าที่**ไม่ขาดหาย** ในแต่ละคอลัมน์ (ถ้าน้อยกว่าจำนวนแถวทั้งหมด แสดงว่ามี missing values!)
- คอลัมน์ `Dtype` → ชนิดข้อมูลของคอลัมน์นั้น (`int64`, `float64`, `object` คือ text)

**สังเกต:** คอลัมน์ `temperature_c`, `humidity_pct`, `rainfall_mm`, `wind_speed_kmh`, `pressure_hpa` มี Non-Null Count **น้อยกว่า 2075** — แสดงว่าข้อมูลจริงนี้มี **missing values** อยู่ ตรงตามที่เกริ่นไว้ตอนต้นคาบ


---
## 7. `describe()` — สถิติพื้นฐาน

`.describe()` คำนวณสถิติเชิงพรรณนา (descriptive statistics) ให้ทุกคอลัมน์ตัวเลขในคำสั่งเดียว — เชื่อมกับฟังก์ชันสถิติของ NumPy ที่เรียนมา (`mean`, `std`, `min`, `max`)


In [ ]:
df.describe()

**อ่านผลลัพธ์ `.describe()` อย่างไร:**

| แถว | ความหมาย |
|---|---|
| `count` | จำนวนค่าที่ไม่ขาดหาย (เทียบกับ Non-Null Count ใน `.info()`) |
| `mean` | ค่าเฉลี่ย |
| `std` | ส่วนเบี่ยงเบนมาตรฐาน (standard deviation) |
| `min` / `max` | ค่าต่ำสุด / สูงสุด |
| `25%`, `50%`, `75%` | เปอร์เซ็นไทล์ (50% คือค่ามัธยฐาน/median) |

**🔍 ลองสังเกตสิ่งผิดปกติ:** ดูที่คอลัมน์ `temperature_c` และ `humidity_pct` —ค่า `max` หรือ `min` ดูสมเหตุสมผลไหม? (จำได้ไหมว่าตอนต้นคาบบอกว่าข้อมูลนี้มี outliers ปนอยู่ — `describe()` มักเป็นจุดแรกที่ช่วยให้สังเกตเห็นความผิดปกติแบบนี้)


In [ ]:
# ดูเฉพาะคอลัมน์ที่น่าสงสัย
print(df[["temperature_c", "humidity_pct", "wind_speed_kmh"]].describe())

### 7.1 `describe()` กับคอลัมน์ที่เป็น text

โดย default `.describe()` จะคำนวณแค่คอลัมน์ตัวเลข — ถ้าอยากดูสถิติของคอลัมน์ text ต้องระบุ `include="object"`


In [ ]:
df.describe(include="object")

**อ่านผลลัพธ์สำหรับคอลัมน์ text:**
- `unique` → จำนวนค่าที่ไม่ซ้ำกัน (เชื่อมกับ `set()` ที่เรียนมา — ถ้าแปลงคอลัมน์เป็น set จะได้จำนวนนี้)
- `top` → ค่าที่ปรากฏบ่อยที่สุด
- `freq` → ความถี่ของค่า `top` นั้น

**🔍 ลองสังเกต:** คอลัมน์ `country` ควรมี unique ประมาณ 23 ประเทศ (จำนวนเมืองในข้อมูล) แต่ถ้าได้มากกว่านั้น อาจเป็นเพราะมีตัวพิมพ์ไม่สม่ำเสมอ (เช่น `"Thailand"` กับ `"THAILAND"` pandas จะมองว่าเป็นค่าต่างกัน!)


---
## 8. `dtypes` — ชนิดข้อมูลของแต่ละคอลัมน์

`.dtypes` แสดงชนิดข้อมูลของทุกคอลัมน์แบบสั้นๆ (ส่วนหนึ่งของสิ่งที่ `.info()` แสดง แต่ดูได้เร็วกว่าถ้าต้องการแค่ส่วนนี้)

| dtype | ความหมาย | เทียบกับชนิดข้อมูล Python ที่เรียนมา |
|---|---|---|
| `int64` | จำนวนเต็ม | `int` |
| `float64` | ทศนิยม | `float` |
| `object` | ข้อความ (หรือข้อมูลผสม) | `str` |
| `bool` | จริง/เท็จ | `bool` |
| `datetime64` | วันที่/เวลา | (ไม่มีใน Python ปกติ ต้องแปลงเอง) |


In [ ]:
print(df.dtypes)

**🔍 จุดสำคัญที่ต้องสังเกต:** คอลัมน์ `date` มี dtype เป็น `object` (text) **ไม่ใช่** `datetime64` — แม้ในไฟล์ CSV จะเป็นรูปแบบวันที่ (`2025-01-01`) แต่ pandas **ไม่แปลงให้อัตโนมัติ** ต้องสั่งแปลงเองด้วย `pd.to_datetime()`


In [ ]:
# แปลงคอลัมน์ date จาก object (text) ให้เป็น datetime จริง
df["date"] = pd.to_datetime(df["date"])
print(df["date"].dtype)
print(df["date"].head())

# ตอนนี้ดึงส่วนประกอบของวันที่ออกมาได้ (เดือน, วัน, วันในสัปดาห์)
print("\nเดือนของ 5 แถวแรก:", df["date"].dt.month.head().tolist())
print("ชื่อวันในสัปดาห์ของ 5 แถวแรก:", df["date"].dt.day_name().head().tolist())

---
## 9. `loc` และ `iloc` — เลือกข้อมูลแบบระบุตำแหน่ง

นี่คือคำสั่งที่ใช้บ่อยที่สุดในการ "เลือกข้อมูลบางส่วน" จาก DataFrame — มี 2 ตัวที่หน้าตาคล้ายกันแต่ทำงานต่างกัน:

| | `.loc[]` | `.iloc[]` |
|---|---|---|
| ย่อมาจาก | **lo**cation (ตาม **label**/ชื่อ) | **i**nteger **loc**ation (ตาม**ตำแหน่ง**) |
| ใช้ระบุแถว | ชื่อ index | ตำแหน่งเป็นเลข (0, 1, 2, ... เหมือน list) |
| ใช้ระบุคอลัมน์ | ชื่อคอลัมน์ | ตำแหน่งเป็นเลข |
| ขอบเขต slicing | **รวม** ค่าท้ายด้วย | **ไม่รวม** ค่าท้าย (เหมือน list/range ที่เรียนมา) |

### 9.1 `.iloc[]` — เลือกตามตำแหน่ง (เหมือน list/array indexing ที่เรียนมา)


In [ ]:
df = pd.read_csv("world_weather_sample.csv")

print("แถวแรก (ตำแหน่ง 0):")
print(df.iloc[0])

print("\n3 แถวแรก:")
print(df.iloc[0:3])    # เหมือน slicing list ที่เรียนมา - ไม่รวมตำแหน่ง 3

print("\nแถวที่ 0, คอลัมน์ที่ 2 (ค่าเดียว):")
print(df.iloc[0, 2])

print("\n3 แถวแรก, 3 คอลัมน์แรก:")
print(df.iloc[0:3, 0:3])

### 9.2 `.loc[]` — เลือกตามชื่อ (label)

เมื่อ DataFrame ใช้ index แบบเลข default (0, 1, 2, ...) `.loc[]` กับ `.iloc[]` ดูคล้ายกัน แต่ต่างกันตรง slicing — `.loc[]` รวมค่าท้ายด้วย!


In [ ]:
print("loc[0:3] -- รวมตำแหน่ง 3 ด้วย (ต่างจาก iloc!):")
print(df.loc[0:3])

print("\nเลือกคอลัมน์ด้วยชื่อผ่าน .loc[]:")
print(df.loc[0:3, "city"])

print("\nเลือกหลายคอลัมน์ด้วยชื่อ:")
print(df.loc[0:3, ["city", "temperature_c", "humidity_pct"]])

### 9.3 `.loc[]` มีประโยชน์มากเมื่อ index ไม่ใช่เลข

ลองตั้ง index ใหม่เป็นชื่อเมือง จะเห็นความแตกต่างของ `.loc[]` ชัดเจนขึ้น


In [ ]:
# ตั้ง city เป็น index แทนเลข 0,1,2,...
df_city_index = df.set_index("city")
print(df_city_index.head())

print("\nเลือกข้อมูลของ Bangkok ทั้งหมดด้วย .loc[] (ระบุชื่อ index ตรงๆ):")
print(df_city_index.loc["Bangkok"].head())

# .iloc[] ยังใช้ตำแหน่งได้เหมือนเดิม ไม่สนใจว่า index ชื่ออะไร
print("\nแถวแรกด้วย .iloc[] (ใช้ตำแหน่งเสมอ ไม่ว่า index จะเป็นอะไร):")
print(df_city_index.iloc[0])

### 9.4 สรุป: เมื่อไหร่ใช้ `.loc[]` เมื่อไหร่ใช้ `.iloc[]`

- ใช้ **`.iloc[]`** เมื่อต้องการอ้างถึง "ตำแหน่งที่เท่าไหร่" (เช่น แถวแรก, 5 แถวสุดท้าย)
- ใช้ **`.loc[]`** เมื่อต้องการอ้างถึง "ชื่อ/label" ที่มีความหมาย (เช่น index เป็นชื่อเมือง, ชื่อคอลัมน์)
- **`.loc[]` ยังใช้กับ boolean indexing ได้ด้วย** (หัวข้อถัดไป) ซึ่งเป็นการใช้งานที่พบบ่อยที่สุดของ `.loc[]` ในทางปฏิบัติ


In [ ]:
# ให้ดึงเฉพาะคอลัมน์ humidity_pct, wind_speed_kmh
df[['humidity_pct', 'wind_speed_kmh']]
df.iloc[:, [6, 8]]
df.loc[:, ['humidity_pct', 'wind_speed_kmh']]

,humidity_pct,wind_speed_kmh
0,72.6,15.9
1,60.6,20.5
2,46.8,8.6
3,73.4,11.8
4,55.0,16.6
...,...,...
2070,63.3,10.5
2071,65.8,13.3
2072,51.0,27.2
2073,65.0,6.9


In [ ]:
# ดึงทุกแถวที่ region = "Asia"
df[df['region'] == 'Asia']
df.loc[df['region'] == 'Asia']

,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
1,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1
3,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1
12,434,2025-03-15,Beijing,China,Asia,14.3,38.5,0.0,13.6,1015.8
24,357,2025-03-28,Singapore,Singapore,Asia,28.5,79.3,3.0,19.6,1013.5
25,334,2025-03-05,Singapore,Singapore,Asia,31.5,150.0,7.0,20.0,1016.3
...,...,...,...,...,...,...,...,...,...,...
2061,245,2025-03-06,Tokyo,Japan,Asia,16.6,62.5,6.5,17.5,1011.9
2064,60,2025-03-01,Bangkok,Thailand,Asia,31.7,82.2,9.2,19.9,1016.8
2066,177,2025-03-28,Chiang Mai,Thailand,Asia,37.3,60.8,3.0,21.3,1017.1
2067,450,2025-03-31,Beijing,China,Asia,17.3,NaN,0.0,13.9,1018.5


In [ ]:
# ดึงค่าในแถวที่ 2,4,6,8,10 คอลัมน์ที่ 3,4,5,6
df.iloc[[2,4,6,8,10],[3,4,5,6]]
df.iloc[2:11:2,3:7]

,country,region,temperature_c,humidity_pct
2,USA,North America,20.6,46.8
4,South Africa,Africa,19.1,55.0
6,Argentina,South America,22.1,63.4
8,Argentina,South America,20.8,58.7
10,Russia,Europe,6.0,70.8


---
## 10. Boolean Indexing — กรองข้อมูลด้วยเงื่อนไข

นี่คือเทคนิคที่ใช้บ่อยที่สุดในการ "กรอง" DataFrame — เหมือน boolean indexing ของ NumPy array ที่เรียนมา แต่ทำกับทั้งตารางได้

### 10.1 เงื่อนไขเดียว


In [92]:
df = pd.read_csv("world_weather_sample.csv")

# สร้าง boolean Series จากเงื่อนไข
is_hot = df["temperature_c"] > 30
print(is_hot.head())    # ได้ True/False ตามแต่ละแถว
print(type(is_hot))      # เป็น Series of bool

print()

# ใช้ boolean Series นั้นกรอง DataFrame (ได้เฉพาะแถวที่ True)
hot_days = df[df["temperature_c"] > 30]
print(f"จำนวนวันที่ร้อนกว่า 30°C: {len(hot_days)} แถว จากทั้งหมด {len(df)} แถว")
print(hot_days.head())

FileNotFoundError: [Errno 2] No such file or directory: 'world_weather_sample.csv'

### 10.2 หลายเงื่อนไขพร้อมกัน (เชื่อมกับ `and`/`or` ที่เรียนมา)

**ข้อสำคัญมาก:** ใน pandas ต้องใช้ `&` (and) และ `|` (or) แทน `and`/`or` ปกติของ Python และ**ต้องครอบแต่ละเงื่อนไขด้วย `()`เสมอ** ไม่อย่างนั้นจะ error


In [ ]:
# เงื่อนไข AND: ร้อนและชื้น
hot_and_humid = df[(df["temperature_c"] > 28) & (df["humidity_pct"] > 70)]
print(f"วันที่ร้อนและชื้น: {len(hot_and_humid)} แถว")
print(hot_and_humid[["city", "temperature_c", "humidity_pct"]].head())

print()

# เงื่อนไข OR: เมืองนี้ หรือเมืองนั้น
bangkok_or_tokyo = df[(df["city"] == "Bangkok") | (df["city"] == "Tokyo")]
print(f"\nข้อมูลของ Bangkok หรือ Tokyo: {len(bangkok_or_tokyo)} แถว")

# ลบ # แล้วลองรันดู จะเห็น error เพราะใช้ and/or ปกติของ Python กับ Series ไม่ได้
# wrong = df[(df["temperature_c"] > 28) and (df["humidity_pct"] > 70)]

วันที่ร้อนและชื้น: 237 แถว
         city  temperature_c  humidity_pct
3      Mumbai           31.0          73.4
18      Lagos           28.8          70.5
21      Lagos           32.8          86.3
24  Singapore           28.5          79.3
25  Singapore           31.5         150.0


ข้อมูลของ Bangkok หรือ Tokyo: 181 แถว


### 10.3 `.isin()` — เช็คว่าค่าอยู่ใน list/set ที่กำหนดไหม

ใช้แทนการเขียน `(df["city"] == "Bangkok") | (df["city"] == "Tokyo") | ...` ยาวๆ เวลามีหลายค่าที่ต้องการเช็ค


In [ ]:
asian_cities = ["Bangkok", "Tokyo", "Singapore", "Beijing", "Mumbai"]   # list ที่เรียนมา

asian_data = df[df["city"].isin(asian_cities)]
print(f"ข้อมูลของเมืองในเอเชีย: {len(asian_data)} แถว")
print(asian_data["city"].unique())   # .unique() ดูว่ามีค่าไม่ซ้ำอะไรบ้าง (เทียบกับ set() ที่เรียนมา)

ข้อมูลของเมืองในเอเชีย: 451 แถว
['Tokyo' 'Mumbai' 'Beijing' 'Singapore' 'Bangkok']


### 10.4 ใช้ `.loc[]` ร่วมกับ boolean indexing (รูปแบบที่ใช้บ่อยที่สุดในงานจริง)

`.loc[]` รับเงื่อนไข boolean ได้ด้วย และยังเลือกคอลัมน์พร้อมกันได้ในคำสั่งเดียว — เป็นรูปแบบที่ใช้บ่อยที่สุด


In [ ]:
# กรองแถว + เลือกคอลัมน์ ในคำสั่งเดียวด้วย .loc[]
result = df.loc[df["temperature_c"] > 35, ["city", "date", "temperature_c"]]
print(result)
print(f"\nพบ {len(result)} แถวที่อุณหภูมิเกิน 35°C (น่าจะเป็นค่าผิดปกติ ลองดูค่าจริงด้านบน!)")

            city        date  temperature_c
78     Cape Town  2025-02-10          999.0
161      Bangkok  2025-02-14           38.6
325       Mumbai  2025-03-31           35.6
343      Bangkok  2025-01-26           37.2
350    Singapore  2025-03-23           35.4
463      Bangkok  2025-03-23           35.5
492   Chiang Mai  2025-03-29           35.6
544       Mumbai  2025-01-12           36.6
712        Paris  2025-03-28          999.0
863      Bangkok  2025-01-28           35.7
877      Bangkok  2025-03-26           39.1
968       Mumbai  2025-03-20           36.4
1042     Bangkok  2025-03-14           35.1
1065       Lagos  2025-03-22           37.2
1104  Chiang Mai  2025-03-27           35.5
1112     Bangkok  2025-03-24           35.7
1152       Lagos  2025-03-31           35.8
1341  Chiang Mai  2025-03-03           37.1
1398     Bangkok  2025-03-31           35.7
1445     Bangkok  2025-03-27           37.4
1620       Lagos  2025-01-14           35.4
1700     Bangkok  2025-02-24    

**🔍 สังเกตจุดสำคัญ:** ผลลัพธ์ข้างบนน่าจะเจอค่า `999.0` ที่เราเกริ่นไว้ตอนต้นคาบว่าเป็น **outlier ที่จงใจใส่ไว้ในข้อมูล** — นี่คือตัวอย่างจริงว่า boolean indexing ใช้ตรวจหาความผิดปกติของข้อมูลได้อย่างไร ซึ่งเป็นขั้นตอนสำคัญก่อนนำข้อมูลไปวิเคราะห์ต่อ (data cleaning จะได้เรียนรายละเอียดเพิ่มในคาบต่อไป)

### 10.5 `~` — กลับค่าเงื่อนไข (NOT)

เหมือน `not` ที่เรียนมาจาก logical operators — ใช้ `~` หน้าเงื่อนไขเพื่อกลับ True เป็น False และ False เป็น True


In [ ]:
# หาแถวที่ "ไม่ใช่" ค่าผิดปกติ (อุณหภูมิอยู่ในช่วงที่สมเหตุสมผล)
normal_temp = df[~(df["temperature_c"] > 50)]
print(f"แถวที่อุณหภูมิปกติ (ไม่เกิน 50°C): {len(normal_temp)} จากทั้งหมด {len(df)}")

# เทียบเท่ากับการเขียนเงื่อนไขตรงข้ามตรงๆ
normal_temp_v2 = df[df["temperature_c"] <= 50]
print("ได้ผลลัพธ์เหมือนกันไหม:", len(normal_temp) == len(normal_temp_v2))

แถวที่อุณหภูมิปกติ (ไม่เกิน 50°C): 2072 จากทั้งหมด 2075
ได้ผลลัพธ์เหมือนกันไหม: False


---
## 🧪 แบบฝึกหัดท้ายคาบ

> ลองทำเองในเซลล์ที่เตรียมไว้ด้านล่างแต่ละข้อ ไม่มีเฉลยให้ — ถ้าไม่แน่ใจให้ลองรันดูผลลัพธ์ หรือถามอาจารย์/เพื่อนได้เลย ทุกข้อใช้ไฟล์ `world_weather_sample.csv`

### ข้อ 1: โหลดและสำรวจข้อมูลเบื้องต้น
1. โหลดไฟล์ `world_weather_sample.csv` เข้าตัวแปร `df`
2. แสดง 10 แถวแรก และ 5 แถวสุดท้าย
3. เรียก `.info()` และ `.describe()` แล้วตอบในคอมเมนต์ (เขียนเป็น `#` ในเซลล์): มีคอลัมน์ไหนบ้างที่มี missing values?


In [83]:
# 1
import pandas as pd
df = pd.read_csv("/content/world_weather_sample-3.csv")
print("โหลดข้อมูลสำเร็จ! ขนาดของข้อมูล:", df.shape)
df

โหลดข้อมูลสำเร็จ! ขนาดของข้อมูล: (2075, 10)


,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
0,983,2025-03-24,New York,USA,North America,15.7,72.6,6.3,15.9,1005.4
1,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1
2,1022,2025-02-01,Los Angeles,USA,North America,20.6,46.8,1.2,8.6,1013.5
3,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1
4,1834,2025-02-03,Cape Town,South Africa,Africa,19.1,55.0,0.0,16.6,1015.1
...,...,...,...,...,...,...,...,...,...,...
2070,1093,2025-01-13,Mexico City,Mexico,North America,17.8,63.3,7.4,10.5,1001.2
2071,1769,2025-02-28,Nairobi,Kenya,Africa,20.6,65.8,3.7,13.3,1015.6
2072,1738,2025-01-28,Nairobi,Kenya,Africa,21.7,51.0,9.5,27.2,1013.9
2073,1210,2025-02-09,Toronto,Canada,North America,NaN,65.0,9.1,6.9,1007.8


In [84]:
#2
df = pd.read_csv("/content/world_weather_sample-3.csv")

print("--- 10 แถวแรก (default) ---")
print(df.head())

print("\n--- 5 แถวสุดท้าย ---")
print(df.tail())

--- 10 แถวแรก (default) ---
   record_id        date         city       country         region  \
0        983  2025-03-24     New York           USA  North America   
1        220  2025-02-09        Tokyo         Japan           Asia   
2       1022  2025-02-01  Los Angeles           USA  North America   
3        469  2025-01-19       Mumbai         India           Asia   
4       1834  2025-02-03    Cape Town  South Africa         Africa   

   temperature_c  humidity_pct  rainfall_mm  wind_speed_kmh  pressure_hpa  
0           15.7          72.6          6.3            15.9        1005.4  
1           18.3          60.6          0.0            20.5        1015.1  
2           20.6          46.8          1.2             8.6        1013.5  
3           31.0          73.4          8.0            11.8        1000.1  
4           19.1          55.0          0.0            16.6        1015.1  

--- 5 แถวสุดท้าย ---
      record_id        date         city country         region  \
2070  

In [85]:
# 3
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075 entries, 0 to 2074
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   record_id       2075 non-null   int64  
 1   date            2075 non-null   object 
 2   city            2075 non-null   object 
 3   country         2075 non-null   object 
 4   region          2075 non-null   object 
 5   temperature_c   2055 non-null   float64
 6   humidity_pct    1993 non-null   float64
 7   rainfall_mm     2013 non-null   float64
 8   wind_speed_kmh  2034 non-null   float64
 9   pressure_hpa    2043 non-null   float64
dtypes: float64(5), int64(1), object(4)
memory usage: 162.2+ KB


,record_id,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
count,2075.000000,2055.000000,1993.000000,2013.000000,2034.000000,2043.000000
mean,1035.148916,21.837421,66.818214,4.147243,14.775418,1013.130739
std,598.021944,38.215806,13.091099,3.852302,5.883939,5.861379
min,1.000000,-88.000000,21.500000,0.000000,-5.000000,993.900000
25%,517.500000,15.400000,58.600000,0.300000,10.800000,1009.300000
50%,1035.000000,20.000000,67.100000,3.500000,15.000000,1013.100000
75%,1553.500000,25.700000,75.500000,6.600000,18.700000,1017.200000
max,2070.000000,999.000000,150.000000,17.600000,33.700000,1030.600000


### ข้อ 2: dtypes และการแปลงชนิดข้อมูล
1. เช็ค `.dtypes` ของ `df` ทั้งหมด
2. แปลงคอลัมน์ `date` เป็น `datetime` ด้วย `pd.to_datetime()`
3. หาว่าข้อมูลนี้มีข้อมูลตั้งแต่วันที่เท่าไหร่ ถึงวันที่เท่าไหร่ (Hint: ใช้ `.min()` และ `.max()` กับคอลัมน์ date ที่แปลงแล้ว)


In [2]:
# เขียนคำตอบข้อ 2 ที่นี่
# 1
import pandas as pd
df = pd.read_csv(
    "/content/world_weather_sample-3.csv"
    )
df.dtypes

,0
record_id,int64
date,object
city,object
country,object
region,object
temperature_c,float64
humidity_pct,float64
rainfall_mm,float64
wind_speed_kmh,float64
pressure_hpa,float64


In [3]:
# 2
df['date'] = pd.to_datetime(df['date'])
df.dtypes

,0
record_id,int64
date,datetime64[ns]
city,object
country,object
region,object
temperature_c,float64
humidity_pct,float64
rainfall_mm,float64
wind_speed_kmh,float64
pressure_hpa,float64


In [4]:
# 3
min_date = df['date'].min()
max_date = df['date'].max()

print("วันที่เริ่มต้น:", min_date)
print("วันทีสิ้นสุด:", max_date)

วันที่เริ่มต้น: 2025-01-01 00:00:00
วันทีสิ้นสุด: 2025-03-31 00:00:00


### ข้อ 3: loc และ iloc
1. ใช้ `.iloc[]` ดึงข้อมูล 5 แถวแรก เฉพาะคอลัมน์ `city`, `temperature_c`, `humidity_pct`
2. ตั้ง index ของ `df` เป็นคอลัมน์ `city` (เก็บไว้ใน DataFrame ใหม่ ไม่ทับของเดิม)
3. ใช้ `.loc[]` ดึงข้อมูลทั้งหมดของเมือง `"Tokyo"` จาก DataFrame ที่ตั้ง index ใหม่แล้ว


In [80]:
df.columns

Index(['record_id', 'date', 'city', 'country', 'region', 'temperature_c',
       'humidity_pct', 'rainfall_mm', 'wind_speed_kmh', 'pressure_hpa'],
      dtype='object')

In [81]:
# 1
df[['city', 'temperature_c', 'humidity_pct']]
df.iloc[:,[2,5,6]]
df.loc[:,['city', 'temperature_c', 'humidity_pct']]

,city,temperature_c,humidity_pct
24,Singapore,28.5,79.3
25,Singapore,31.5,150.0
29,Singapore,30.0,80.5
46,Singapore,33.3,84.7
67,Bangkok,30.8,74.5
...,...,...,...
2004,Singapore,30.8,67.1
2031,Bangkok,33.1,68.3
2039,Singapore,36.1,85.9
2064,Bangkok,31.7,82.2


In [96]:
# 2
df = pd.read_csv("/content/world_weather_sample-3.csv")
df_city = df.set_index("city")

print(df_city.head())

             record_id        date       country         region  \
city                                                              
New York           983  2025-03-24           USA  North America   
Tokyo              220  2025-02-09         Japan           Asia   
Los Angeles       1022  2025-02-01           USA  North America   
Mumbai             469  2025-01-19         India           Asia   
Cape Town         1834  2025-02-03  South Africa         Africa   

             temperature_c  humidity_pct  rainfall_mm  wind_speed_kmh  \
city                                                                    
New York              15.7          72.6          6.3            15.9   
Tokyo                 18.3          60.6          0.0            20.5   
Los Angeles           20.6          46.8          1.2             8.6   
Mumbai                31.0          73.4          8.0            11.8   
Cape Town             19.1          55.0          0.0            16.6   

             press

In [94]:
# 3
df = pd.read_csv("/content/world_weather_sample-3.csv")
df_city_index = df.set_index("city")

print(df_city_index.loc["Tokyo"].head())

       record_id        date country region  temperature_c  humidity_pct  \
city                                                                       
Tokyo        220  2025-02-09   Japan   Asia           18.3          60.6   
Tokyo        260  2025-03-21   Japan   Asia           19.7          67.4   
Tokyo        262  2025-03-23   Japan   Asia           17.7          60.2   
Tokyo        203  2025-01-23   Japan   Asia           20.8          61.6   
Tokyo        218  2025-02-07   Japan   Asia           19.5          68.2   

       rainfall_mm  wind_speed_kmh  pressure_hpa  
city                                              
Tokyo          0.0            20.5        1015.1  
Tokyo          2.8            16.6        1014.9  
Tokyo          4.0            21.5        1015.8  
Tokyo         12.3            10.9        1023.4  
Tokyo          5.0            13.4        1011.8  


### ข้อ 4: Boolean Indexing — หาความผิดปกติ
จากที่รู้ว่าข้อมูลนี้มี outliers ปนอยู่ จงเขียนโค้ดหา:
1. แถวที่ `humidity_pct` มากกว่า 100 (เป็นไปไม่ได้ในความจริง เพราะความชื้นสูงสุดคือ 100%)
2. แถวที่ `wind_speed_kmh` น้อยกว่า 0 (เป็นไปไม่ได้ เพราะความเร็วลมติดลบไม่ได้)
3. รวมทั้ง 2 เงื่อนไขเข้าด้วยกันด้วย `|` แล้วนับว่าพบกี่แถวทั้งหมด


In [98]:
# 1
df = pd.read_csv("/content/world_weather_sample-3.csv")
df[df["wind_speed_kmh"] < 0]

,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
805,1346,2025-03-27,Sao Paulo,Brazil,South America,27.0,65.5,4.1,-5.0,1023.7
1876,445,2025-03-26,Beijing,China,Asia,16.1,39.8,5.0,-5.0,1009.1


In [99]:
# 2
df = pd.read_csv("/content/world_weather_sample-3.csv")
df[(df["humidity_pct"] > 100) | (df["wind_speed_kmh"] < 0)]

,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
25,334,2025-03-05,Singapore,Singapore,Asia,31.5,150.0,7.0,20.0,1016.3
37,1017,2025-01-27,Los Angeles,USA,North America,17.3,150.0,1.3,8.6,1018.7
805,1346,2025-03-27,Sao Paulo,Brazil,South America,27.0,65.5,4.1,-5.0,1023.7
1876,445,2025-03-26,Beijing,China,Asia,16.1,39.8,5.0,-5.0,1009.1
2068,56,2025-02-25,Bangkok,Thailand,Asia,34.4,150.0,5.6,15.4,1010.7


In [7]:
# 3
df = pd.read_csv("/content/world_weather_sample-3.csv")
df[(df["humidity_pct"] > 100) | (df["wind_speed_kmh"] < 0)]

,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
25,334,2025-03-05,Singapore,Singapore,Asia,31.5,150.0,7.0,20.0,1016.3
37,1017,2025-01-27,Los Angeles,USA,North America,17.3,150.0,1.3,8.6,1018.7
805,1346,2025-03-27,Sao Paulo,Brazil,South America,27.0,65.5,4.1,-5.0,1023.7
1876,445,2025-03-26,Beijing,China,Asia,16.1,39.8,5.0,-5.0,1009.1
2068,56,2025-02-25,Bangkok,Thailand,Asia,34.4,150.0,5.6,15.4,1010.7


### ข้อ 5: ท้าทายเพิ่มเติม (Challenge) — รวมทุกอย่างเข้าด้วยกัน
จงเขียนโค้ดที่:
1. กรองหาข้อมูลเฉพาะ `region == "Asia"` และ `rainfall_mm > 5`
2. จากผลลัพธ์ข้อ 1 เลือกแค่คอลัมน์ `city`, `date`, `rainfall_mm` ด้วย `.loc[]`
3. ใช้ `.isin()` กรองต่อให้เหลือเฉพาะข้อมูลของ `["Bangkok", "Singapore"]`
4. นับจำนวนแถวที่ได้ในที่สุด และลอง print เมืองที่ไม่ซ้ำกัน (`.unique()`) เพื่อตรวจสอบว่ากรองถูกต้อง


In [ ]:
# 1
df1 = df[(df['region'] == 'Asia')&(df['rainfall_mm'] > 5)]
df1

,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
25,334,2025-03-05,Singapore,Singapore,Asia,31.5,150.0,7.0,20.0,1016.3
29,314,2025-02-13,Singapore,Singapore,Asia,30.0,80.5,11.1,5.6,1009.8
46,305,2025-02-04,Singapore,Singapore,Asia,33.3,84.7,8.1,9.2,1010.8
79,10,2025-01-10,Bangkok,Thailand,Asia,28.9,71.3,12.2,17.1,1002.4
80,282,2025-01-12,Singapore,Singapore,Asia,26.9,68.6,15.0,13.1,1011.5
...,...,...,...,...,...,...,...,...,...,...
2004,316,2025-02-15,Singapore,Singapore,Asia,30.8,67.1,13.7,10.6,1008.1
2031,12,2025-01-12,Bangkok,Thailand,Asia,33.1,68.3,6.8,17.0,1018.9
2039,353,2025-03-24,Singapore,Singapore,Asia,36.1,85.9,9.3,5.4,1011.5
2064,60,2025-03-01,Bangkok,Thailand,Asia,31.7,82.2,9.2,19.9,1016.8


In [ ]:
# 2
df.loc[:, ['city', 'date', 'rainfall_mm']]

,city,date,rainfall_mm
0,New York,2025-03-24,6.3
1,Tokyo,2025-02-09,0.0
2,Los Angeles,2025-02-01,1.2
3,Mumbai,2025-01-19,8.0
4,Cape Town,2025-02-03,0.0
...,...,...,...
2070,Mexico City,2025-01-13,7.4
2071,Nairobi,2025-02-28,3.7
2072,Nairobi,2025-01-28,9.5
2073,Toronto,2025-02-09,9.1


In [90]:
# 3
df = df[df['city'].isin(['Bangkok', 'Singapore'])]
print(df)

      record_id        date       city    country region  temperature_c  \
24          357  2025-03-28  Singapore  Singapore   Asia           28.5   
25          334  2025-03-05  Singapore  Singapore   Asia           31.5   
29          314  2025-02-13  Singapore  Singapore   Asia           30.0   
46          305  2025-02-04  Singapore  Singapore   Asia           33.3   
67           53  2025-02-22    Bangkok   Thailand   Asia           30.8   
...         ...         ...        ...        ...    ...            ...   
2004        316  2025-02-15  Singapore  Singapore   Asia           30.8   
2031         12  2025-01-12    Bangkok   Thailand   Asia           33.1   
2039        353  2025-03-24  Singapore  Singapore   Asia           36.1   
2064         60  2025-03-01    Bangkok   Thailand   Asia           31.7   
2068         56  2025-02-25    Bangkok   Thailand   Asia           34.4   

      humidity_pct  rainfall_mm  wind_speed_kmh  pressure_hpa  
24            79.3          3.0    

In [10]:
# 4
result = df[df['city'].isin(['Singapore', 'Bangkok'])]

print('จำนวนแถวสุดท้าย:', len(result))
print('เมืองที่ไม่ซ้ำ:', result['city'].unique())

จำนวนแถวสุดท้าย: 181
เมืองที่ไม่ซ้ำ: ['Singapore' 'Bangkok']


---
## 🔗 เชื่อม Colab กับ GitHub

เก็บ Notebook นี้ (พร้อมไฟล์ `world_weather_sample.csv`) ขึ้น GitHub ต่อจากไฟล์ก่อนหน้า

### วิธีที่ 1: เซฟจาก Colab ขึ้น GitHub ตรงๆ (สำหรับ Notebook)

1. ใน Colab ไปที่เมนู **File → Save a copy in GitHub**
2. เลือก repository เดิมที่ใช้เก็บ Notebook คาบก่อนๆ (เช่น `SC612104-coursework`)
3. ตั้งชื่อไฟล์และ commit message (เช่น `"เพิ่ม notebook: pandas Series/DataFrame/read_csv"`) แล้วกด **OK**

**⚠️ ข้อควรรู้:** วิธีนี้ push แค่ตัว `.ipynb` แต่**ไม่ได้ push ไฟล์ CSV ไปด้วย** — ต้องอัปโหลดไฟล์ CSV ขึ้น GitHub แยกต่างหาก (ผ่านเว็บ GitHub โดยตรง หรือ `git add` ผ่าน Terminal)

### วิธีที่ 2: ใช้ Git ผ่าน Terminal (push ได้ทั้ง Notebook และ CSV)

```bash
git clone https://github.com/<username>/<repo-name>.git
cd <repo-name>
git add notebook.ipynb world_weather_sample.csv
git commit -m "เพิ่ม notebook และข้อมูล: pandas basics"
git push origin main
```

> 💡 **Tip:** ถ้าไฟล์ CSV มีขนาดใหญ่มาก (หลาย MB) ควรพิจารณาใช้ Git LFS หรือเก็บไฟล์ไว้ที่อื่น (เช่น Google Drive) แล้วโหลดผ่าน URL แทน — แต่สำหรับไฟล์ขนาดเล็กแบบนี้ (`world_weather_sample.csv` ~135 KB) push ได้เลย
